# Exploring the Polimi study regulations

This notebook tests how to turn part of the 2026/27 study regulations into
searchable evidence for the Enterprise Knowledge Agent. It focuses on four
courses from the T2D table on PDF page 13.

We compared the extracted credits with the original PDF. Reading each credit
cell directly recovered values such as 10.0 that the initial table extraction
returned incorrectly. We kept credit annotations as text for later review.

Each course record keeps its title, credits, track, academic year, and PDF page.
A simple title search checks that we can find Strategy & Marketing, retrieve
its 10.0 CFU, and point back to page 13.

### Why we checked the credit values

The PDF shows `10.0` CFU for Accounting, Finance & Control (course `096078`).
PyMuPDF's table extraction found the correct row and column, but returned
`100\n.` for the credit cell. The decimal point came after the digits on a
separate line in the extracted text. Removing the line break would have
produced `100.` and left the value wrong.

We fixed this by using the detected cell's boundaries to read just that part
of the page. We passed `sort=True` so PyMuPDF used the text's position on the
page to determine its reading order. This returned `10.0`, which matched the
original PDF. We checked three more course credit values on page 13 and
confirmed that they matched too.

In [ ]:
from pathlib import Path
import pymupdf

print(Path.cwd())

Count the pages.

In [2]:
pdf_path = Path("..") / "Course of Study Regulations.pdf"
pdf = pymupdf.open(pdf_path)

print(f"Total pages: {len(pdf)}")

Total pages: 50


Preview First Page

In [3]:
first_page = pdf[0]
page_text = first_page.get_text()

print(page_text[:500])

Academic Year 2026/27 School of Industrial and Information
Engineering Teaching Regulations for the Degree Course in:
Computer Science and Engineering Master's Degree Location: Milan
1. General Information
School
School of Industrial and Information Engineering
Course of Study Code
542
Course of Study
Computer Science and Engineering
Ordering
Order Ministerial Decree 1648-1649/23
Graduation Class
LM-32 - Computer Engineering
Level
Master's Degree
First AA of activation
2010/2011
Nominal duration


Read a page with course tables

In [4]:
pdf_page_number = 13
page_index = pdf_page_number - 1

course_table_page = pdf[page_index]
extracted_text = course_table_page.get_text()

print(f"Extracted text from PDF page {pdf_page_number}")
print(extracted_text)

Extracted text from PDF page 13
Code Training
activities
SSD
SM
639/24
Teaching Name
Tongue Period CFU
CFU
Group
--
--
--
Courses chosen from the
SWT TABLE Group -
INTER 
(c)
--
--
--
089254
--
--
FINAL EXAM (INF)
--
1st
semester
20.0
20.0
(a) 
Courses reserved for students enrolled in the Enhance program
(b) 
Students who intend to join the SWT initiative must select at least 15 credits from this group
(c) 
Courses reserved for students enrolled in the SWT program - select at least 15 credits
2nd Year Courses - Previously Approved Study Plan: T2D - ICT ENGINEERING,
BUSINESS AND INNOVATION
Code Training
activities
SSD
SM
639/24
Teaching Name
Tongue Period
CFU
CFU
Group
096078
C
IEGE-
01/A
ACCOUNTING,
FINANCE & CONTROL
1st
semester
10.0
10.0
052795
C
IEGE-
01/A
STRATEGY &
MARKETING
1st
semester
10.0
[1.0 
]
10.0
057028
C
IEGE-
01/A
DESIGN THINKING
FOR BUSINESS
1st
semester
5.0
10.0
057108
C
IEGE-
01/A
REGULATION OF
DIGITAL INNOVATION
ON PAYMENTS AND
FINANCE 
(a)
1st
semester
5.0
056993


Preview the tables on page 13

In [5]:
detected_tables = course_table_page.find_tables()
tables = detected_tables.tables

print(f"Tables found on PDF page {pdf_page_number}: {len(tables)}")

for table_number, table in enumerate(tables, start=1):
    rows = table.extract()
    preview_rows = rows[:5]

    print(f"\nTable {table_number} — first five rows")
    for row in preview_rows:
        print(row)

Tables found on PDF page 13: 2

Table 1 — first five rows
['Code', 'Training\nactivities', 'SSD\nSM\n639/24', 'TeachingName', 'Tongue', 'Period', 'CFU', 'CFU\nGroup']
['--', '--', '--', 'Courseschosenfromthe\nSWT TABLE Group\n-\n(c)\nINTER', '--', '--', '--', '']
['089254', '--', '--', 'FINALEXAM(INF)', '--', '1st\nsemester', '200\n.', '200\n.']

Table 2 — first five rows
['Code', 'Training\nactivities', 'SSD\nSM\n639/24', 'TeachingName', 'Tongue', 'Period', 'CFU', 'CFU\nGroup']
['096078', 'C', 'IEGE\n-\n01/A', 'ACCOUNTING\n,\nFINANCE&CONTROL', '', '1st\nsemester', '100\n.', '100\n.']
['052795', 'C', 'IEGE\n-\n01/A', 'STRATEGY&\nMARKETING', '', '1st\nsemester', '100\n.\n[10 ]\n.', '100\n.']
['057028', 'C', 'IEGE\n-\n01/A', 'DESIGNTHINKING\nFORBUSINESS', '', '1st\nsemester', '50\n.', '100\n.']
['057108', 'C', 'IEGE\n-\n01/A', 'REGULATIONOF\nDIGITALINNOVATION\nONPAYMENTSAND\n(a)\nFINANCE', '', '1st\nsemester', '50\n.', None]


6. 1 column at a time.

In [6]:
course_table = tables[1]
course_rows = course_table.extract()

column_headers = course_rows[0]
first_course_row = course_rows[1]

for column_header, cell_value in zip(column_headers, first_course_row):
    readable_header = column_header.replace("\n", " ")
    print(f"{readable_header}: {cell_value!r}")

Code: '096078'
Training activities: 'C'
SSD SM 639/24: 'IEGE\n-\n01/A'
TeachingName: 'ACCOUNTING\n,\nFINANCE&CONTROL'
Tongue: ''
Period: '1st\nsemester'
CFU: '100\n.'
CFU Group: '100\n.'


7.

Issue extracting CFU so only checking CFU column.

In [7]:
from pymupdf import Rect

first_course_table_row = course_table.rows[1]
cfu_column_index = 6
cfu_cell_bounds = Rect(first_course_table_row.cells[cfu_column_index])

cfu_cell_text = course_table_page.get_text(
    clip=cfu_cell_bounds,
    sort=True,
)

print("Expected CFU from the PDF: 10.0")
print(f"Extracted CFU: {cfu_cell_text!r}")

Expected CFU from the PDF: 10.0
Extracted CFU: '10.0'


8. Check 3 credit values from pdf

In [8]:
expected_cfu_by_course = {
    "096078": "10.0",
    "057028": "5.0",
    "057108": "5.0",
}
cfu_column_index = 6

for row_index, course_row in enumerate(course_rows[1:], start=1):
    course_code = course_row[0]
    if course_code not in expected_cfu_by_course:
        continue

    table_row = course_table.rows[row_index]
    cfu_cell_bounds = Rect(table_row.cells[cfu_column_index])
    extracted_cfu = course_table_page.get_text(
        clip=cfu_cell_bounds,
        sort=True,
    ).strip()

    expected_cfu = expected_cfu_by_course[course_code]
    if extracted_cfu == expected_cfu:
        print(f"{course_code}: {extracted_cfu} CFU matches the PDF")
    else:
        print(f"{course_code}: expected {expected_cfu}, extracted {extracted_cfu!r}")

096078: 10.0 CFU matches the PDF
057028: 5.0 CFU matches the PDF
057108: 5.0 CFU matches the PDF


9. Inspect a credit cell with an annotation

splitlines() lets you see what PyMuPDF read on each line. Compare the output with the screenshot. If 10.0 comes through on its own line, we can try storing the main credit value and the annotation separately. Keep the annotation as text until you’ve checked what the symbol means in the regulations.

In [9]:
target_course_code = "052795"
cfu_column_index = 6  # CFU is the seventh column. Python starts counting at 0.

# Skip the header and start at 1 to match the row positions in course_table.
for row_index, course_row in enumerate(course_rows[1:], start=1):
    course_code = course_row[0]
    if course_code != target_course_code:
        continue

    # Use the CFU cell's boundaries to read just that part of the PDF page.
    table_row = course_table.rows[row_index]
    cfu_cell_bounds = Rect(table_row.cells[cfu_column_index])
    cfu_cell_text = course_table_page.get_text(
        clip=cfu_cell_bounds,
        sort=True,
    )

    # splitlines() separates the extracted text at each line break.
    # We want to see whether 10.0 and the bracketed note are on separate lines.
    cfu_lines = cfu_cell_text.splitlines()

    print(f"Course code: {course_code}")
    print("CFU cell, line by line:")

    # Keep the bracketed note as text until we verify its meaning in the PDF.
    for line_number, line_text in enumerate(cfu_lines, start=1):
        print(f"Line {line_number}: {line_text}")

    break  # We found the course we wanted, so stop searching.

Course code: 052795
CFU cell, line by line:
Line 1:   10.0
Line 2: [1.0      ]


10. Store one course’s credit information

In [10]:
# Remove spaces around the credit value on the first line.
main_credit_text = cfu_lines[0].strip()

# Keep the remaining lines together as the annotation.
# strip() removes spaces at the ends and leaves the spacing inside the brackets.
credit_annotation_text = "\n".join(cfu_lines[1:]).strip()

# Store the extracted values with the PDF page where we checked them.
course_credit_record = {
    "course_code": target_course_code,
    "cfu": main_credit_text,
    "cfu_annotation_text": credit_annotation_text,
    "pdf_page": pdf_page_number,
}

for field_name, field_value in course_credit_record.items():
    print(f"{field_name}: {field_value}")

course_code: 052795
cfu: 10.0
cfu_annotation_text: [1.0      ]
pdf_page: 13


11. Build a small set of searchable course records

In [11]:
checked_course_codes = {"096078", "052795", "057028", "057108"}
course_records = []

title_column_index = 3
cfu_column_index = 6

# Skip the header and keep the row index aligned with the detected table.
for row_index, course_row in enumerate(course_rows[1:], start=1):
    course_code = course_row[0].strip()
    if course_code not in checked_course_codes:
        continue

    table_row = course_table.rows[row_index]

    # Read the title from its own cell and join any wrapped lines with spaces.
    title_cell_bounds = Rect(table_row.cells[title_column_index])
    title_cell_text = course_table_page.get_text(
        clip=title_cell_bounds,
        sort=True,
    )
    course_title = " ".join(title_cell_text.split())

    # In these checked rows, the first line is the main credit value.
    # Keep any later lines as an annotation without interpreting them.
    cfu_cell_bounds = Rect(table_row.cells[cfu_column_index])
    cfu_cell_text = course_table_page.get_text(
        clip=cfu_cell_bounds,
        sort=True,
    )
    cfu_lines = cfu_cell_text.strip().splitlines()
    main_credit_text = cfu_lines[0].strip()
    credit_annotation_text = "\n".join(cfu_lines[1:]).strip()

    course_record = {
        "course_code": course_code,
        "course_title": course_title,
        "cfu": main_credit_text,
        "cfu_annotation_text": credit_annotation_text,
        "track": "T2D",
        "academic_year": "2026/27",
        "pdf_page": pdf_page_number,
    }
    course_records.append(course_record)

# Print the key fields so we can check each record before searching.
for course_record in course_records:
    course_code = course_record["course_code"]
    course_title = course_record["course_title"]
    course_credits = course_record["cfu"]
    print(f"{course_code} | {course_title} | {course_credits} CFU")

096078 | ACCOUNTING, FINANCE & CONTROL | 10.0 CFU
052795 | STRATEGY & MARKETING | 10.0 CFU
057028 | DESIGN THINKING FOR BUSINESS | 5.0 CFU
057108 | REGULATION OF DIGITAL INNOVATION ON PAYMENTS AND (a) FINANCE | 5.0 CFU


12. Find a course by title.

In [12]:
search_title = "Strategy & Marketing"
normalized_search_title = search_title.casefold()

matching_records = []

for course_record in course_records:
    course_title = course_record["course_title"]

    # casefold() lets us compare titles regardless of capitalization.
    if normalized_search_title in course_title.casefold():
        matching_records.append(course_record)

if not matching_records:
    print(f"No course found for: {search_title}")

for matching_record in matching_records:
    course_title = matching_record["course_title"]
    course_credits = matching_record["cfu"]
    track = matching_record["track"]
    academic_year = matching_record["academic_year"]
    source_page = matching_record["pdf_page"]

    # Show the retrieved evidence and where to verify it in the PDF.
    print(f"{course_title} is worth {course_credits} CFU.")
    print(f"Source: {track}, {academic_year} regulations, PDF page {source_page}.")

STRATEGY & MARKETING is worth 10.0 CFU.
Source: T2D, 2026/27 regulations, PDF page 13.


### Verified result

A title lookup for Strategy & Marketing retrieved course `052795`, its
10.0 CFU value, and its source: the 2026/27 T2D regulations, PDF page 13.

#### Questions Testing

13. Locate and verify the T2A rule in the PDF that requires at least 45 CFU from TABA and at least 55 CFU across TABA and TABB combined, so the assistant can retrieve and cite it.

In [13]:
search_phrase = "almeno 55 cfu"

# Start counting at 1 so the result matches the PDF viewer's page number.
for pdf_page_number, page in enumerate(pdf, start=1):
    page_text = page.get_text(sort=True)

    # split() breaks the text into words. Joining them with spaces
    # lets us match the phrase even if the PDF wraps it onto two lines.
    searchable_text = " ".join(page_text.split()).casefold()

    if search_phrase in searchable_text:
        print(f"Found the phrase on PDF page {pdf_page_number}.")
        print(page_text)

Found the phrase on PDF page 40.
11/12/26, 1:11 PM                                                Course of Study Regulations
      Una descrizione dettagliata dei percorsi e dei rispettivi insegnamenti suggeriti è disponibile sul sito
       web: https://www.inginformatica.polimi.it/cosa-studio-magistrale/ .

        Attività a scelta dello studente e piani autonomi degli studi

      Lo studente può proporre un proprio piano autonomo degli studi che differisca dal piano T2A per
      un massimo di 20 crediti di insegnamenti. Il Consiglio di Corsi di Studi si riserva di approvare o
         rigettare tali piani autonomi in base alla loro coerenza con l'offerta formativa e con il percorso
       complessivo proposto dallo studente.
        In caso di scelta di insegnamenti che non compaiono nelle tabelle del presente regolamento, si
        consiglia di contattare preventivamente la Commissione Piani di Studio per un parere, in modo
        rendere più semplice e spedita la verifica de

14. TABA and TABB rule record

### Extracting the TABA and TABB credit rule

We extracted this rule from page 40 of the 2026/27 regulations so the
assistant can find and cite it when a student asks how many TABA courses
they need to take in T2A.

The rule sets credit requirements: at least 45 CFU from TABA and at least
55 CFU across TABA and TABB combined. We keep both thresholds in the
same searchable record so the assistant can explain how they fit together.
The record preserves the original Italian wording and the PDF page number.

Q08 in `evaluation/development_questions.csv` holds the question and the
answer we verified by hand. First, we’ll use BM25 to check whether the
question retrieves the rule from page 40. Then we’ll use RAG to check
whether the language model can explain the rule accurately in English
and cite that page.

In [14]:
rule_page_number = 40
rule_page = pdf[rule_page_number - 1]
rule_page_text = rule_page.get_text(sort=True)

# Join wrapped lines so we can find the rule as one continuous sentence.
normalized_page_text = " ".join(rule_page_text.split())
rule_start = normalized_page_text.casefold().find("almeno 55 cfu")

if rule_start == -1:
    print("Could not find the start of the TABA/TABB rule.")
else:
    # In this PDF, the semicolon marks the end of this numbered requirement.
    rule_end = normalized_page_text.find(";", rule_start)

    if rule_end == -1:
        print("Could not find the end of the rule. Check page 40 manually.")
    else:
        rule_text = normalized_page_text[rule_start:rule_end].strip()

        # Keep the source details with the text so we can cite it after retrieval.
        study_plan_rule = {
            "section": "Vincoli addizionali per il piano di studi T2A",
            "track": "T2A",
            "academic_year": "2026/27",
            "pdf_page": rule_page_number,
            "text": rule_text,
        }

        print(f"PDF page: {study_plan_rule['pdf_page']}")
        print(study_plan_rule["text"])

PDF page: 40
almeno 55 cfu complessivi nei Gruppi TABA e TABB, di cui almeno 45 cfu dal gruppo TABA


### Testing BM25 keyword search

We use BM25 to rank the five source records and check whether two
development questions retrieve the correct PDF pages. This tests
retrieval before we add a language model to generate answers.

1. Prepare the searchable words

We combine four course records and the TABA/TABB rule into one list
called `search_records`. Each course becomes a short sentence with its
code, title, and credits. The rule keeps its original Italian wording.

Each record also stores its track, academic year, and PDF page so we
can tell which study plan it applies to and verify it in the regulations.
For example, the Strategy & Marketing record contains 10.0 CFU and
points back to page 13.

The final count should be five records. The next cell uses their text
and track to build the BM25 index.

In [15]:
search_records = []

# Turn each course into searchable text and keep its source details.
for course_record in course_records:
    course_code = course_record["course_code"]
    course_title = course_record["course_title"]
    course_credits = course_record["cfu"]

    course_text = f"Course {course_code}: {course_title}. Credits: {course_credits} CFU."

    search_record = {
        "record_type": "course",
        "text": course_text,
        "track": course_record["track"],
        "academic_year": course_record["academic_year"],
        "pdf_page": course_record["pdf_page"],
    }
    search_records.append(search_record)

# Add the page 40 rule so BM25 can search it alongside the courses.
rule_search_record = {
    "record_type": "rule",
    "text": study_plan_rule["text"],
    "track": study_plan_rule["track"],
    "academic_year": study_plan_rule["academic_year"],
    "pdf_page": study_plan_rule["pdf_page"],
}
search_records.append(rule_search_record)

print(f"Total searchable records: {len(search_records)}")

Total searchable records: 5


### BM25 keyword search

BM25 is a keyword search method that ranks records by how closely
their words match a question. It gives more weight to distinctive
terms such as `TABA` and gives repeated words diminishing weight.
It also accounts for differences in record length.

In this notebook, BM25 searches five records: four course rows and one
study-plan rule. We split the questions and records into words using the
same rules, then inspect the highest-ranked result for each question.

Both test questions retrieved the expected evidence:

- **Strategy & Marketing:** BM25 found course `052795` in T2D, worth
  10.0 CFU, on PDF page 13.
- **TABA in T2A:** BM25 found the Italian rule on PDF page 40. It sets
  a minimum of 45 CFU from TABA and 55 CFU across TABA and TABB combined.

We use BM25 as our first retrieval method because we can inspect what
it finds before adding a language model. The outputs above are retrieved
evidence. In the next stage, RAG will use that evidence to write an
answer and cite the relevant PDF page. Later, we can compare BM25 with
semantic search using the same questions.

In [16]:
import re

from rank_bm25 import BM25Okapi


def tokenize_text(text):
    # Lowercase the text and extract words and numbers.
    # For example, "TABA and TABB?" becomes ["taba", "and", "tabb"].
    return re.findall(r"\w+", text.casefold())


tokenized_records = []

for search_record in search_records:
    # Include the track so a question mentioning T2A or T2D can match it.
    searchable_text = f"{search_record['track']} {search_record['text']}"
    record_tokens = tokenize_text(searchable_text)
    tokenized_records.append(record_tokens)

bm25_index = BM25Okapi(tokenized_records)

print(f"BM25 indexed {len(tokenized_records)} records.")

BM25 indexed 5 records.


In [17]:
test_questions = [
    "How many credits is Strategy & Marketing in the second-year T2D plan?",
    "How many TABA courses must I take in T2A?",
]

for question in test_questions:
    # Use the same word-splitting rules for questions and source records.
    question_tokens = tokenize_text(question)

    # BM25 returns the highest-ranked record for this question.
    top_results = bm25_index.get_top_n(question_tokens, search_records, n=1)
    top_record = top_results[0]

    print(f"\nQuestion: {question}")
    print(f"Top result: {top_record['track']}, PDF page {top_record['pdf_page']}")
    print(top_record["text"])


Question: How many credits is Strategy & Marketing in the second-year T2D plan?
Top result: T2D, PDF page 13
Course 052795: STRATEGY & MARKETING. Credits: 10.0 CFU.

Question: How many TABA courses must I take in T2A?
Top result: T2A, PDF page 40
almeno 55 cfu complessivi nei Gruppi TABA e TABB, di cui almeno 45 cfu dal gruppo TABA


### What we verified today

We built a small BM25 index with five records: four course rows from
PDF page 13 and one T2A study-plan rule from page 40.

Both questions retrieved the expected record in first place:

- Strategy & Marketing: course `052795`, 10.0 CFU, PDF page 13.
- TABA in T2A: at least 45 CFU from TABA and at least 55 CFU across
  TABA and TABB combined, PDF page 40.

These are two retrieval checks on a small sample. Our current search
code always asks for the top result, even when that result may be
irrelevant. We still need to test how it handles questions with
insufficient evidence.

### 19th November Session: First cited RAG answer

We will start with Q08. We will pass the English question, the
retrieved Italian passage, and its PDF page number to a language model.
Then we will check whether the answer preserves both credit thresholds
and cites page 40.

Once that works, we will test a question the current collection cannot
answer. That case will help us decide when the agent should retry the
search or explain that it lacks enough evidence.